# RamanV2 光谱查看器

顶部 `INPUT_PATH` 填单个 `.arc_data` 文件或一个文件夹，然后依次运行准备数据单元和绘图单元。
文件夹输入会按 `SAMPLE_SEED` 稳定抽一条谱画单谱，并额外画整个文件夹。

In [ ]:
"""RamanV2 光谱查看脚本。
顶部填数据集与一个输入路径（单个 .arc_data 或文件夹），然后依次运行准备和绘图单元。
"""

import random
from pathlib import Path

from ramanv2.analysis.spectrum_viewer import (
    collect_folder_spectra,
    plot_baseline_comparison,
    plot_baseline_fit,
    plot_folder_spectra,
    plot_raw_spectrum,
    prepare_single_spectrum,
)
from ramanv2.common.paths import PROJECT_ROOT
from ramanv2.common.plotting import configure_matplotlib_fonts
from ramanv2.core.config import build_config
from ramanv2.data.config import resolve_build_config
from ramanv2.data.profiles import resolve_selection


# ------------------------------ 顶部运行配置 ------------------------------
# 数据集：沿用它的 bad_bands 与输入规格，和训练/推理链路一致。
DATASET_ID = "PB"
# 单个 .arc_data 文件，或包含 .arc_data 的文件夹；相对路径基于项目根。
INPUT_PATH = r""
# 输入是文件夹时，用它稳定抽取一条谱。
SAMPLE_SEED = 46
# 显示区间；None 表示整条谱。
BASELINE_DISPLAY_MIN = None
BASELINE_DISPLAY_MAX = None
FOLDER_DISPLAY_MIN = None
FOLDER_DISPLAY_MAX = None

## 准备单条谱数据

先运行这一块，后面五个绘图块可以分别运行。

In [ ]:
configure_matplotlib_fonts()
input_path = Path(INPUT_PATH)
if not input_path.is_absolute():
    input_path = PROJECT_ROOT / input_path
if input_path.is_dir():
    folder_dir = input_path
    sample_path = random.Random(SAMPLE_SEED).choice(collect_folder_spectra(folder_dir))
elif input_path.is_file():
    sample_path = input_path
    folder_dir = None
else:
    raise FileNotFoundError(f"输入路径不存在：{input_path}")

# 用数据集规格构建输入配置（含 dataset.json 的坏区），与训练/推理链路一致。
selection = resolve_selection(DATASET_ID, None)
spec = selection.spec
profile = selection.profile
input_config = build_config(
    {
        "dataset_id": spec.dataset_id,
        "profile_id": profile.profile_id,
        "label_schema": selection.hierarchy.label_schema,
        "bad_bands": spec.bad_bands,
    }
).input
build_values = resolve_build_config()
baseline_xlim = None
if BASELINE_DISPLAY_MIN is not None and BASELINE_DISPLAY_MAX is not None:
    baseline_xlim = (BASELINE_DISPLAY_MIN, BASELINE_DISPLAY_MAX)
prepared = prepare_single_spectrum(
    sample_path, input_config, build_values
)
print(f"输入：{input_path}")
print(f"数据集：{spec.dataset_id} / {spec.display_name}")
print(f"读取光谱：{prepared.path}")
print(f"坏区：{input_config.bad_bands}")
print(
    f"基线方法：{build_values.baseline_method}"
    f"（拟合区间 {build_values.baseline_fit_min:g}-{build_values.baseline_fit_max:g}）"
)

## 单条原始谱


In [ ]:
plot_raw_spectrum(prepared)


## 原始谱与计算基线

蓝线为原始谱，橙线为本次实际计算并扣除的基线；两者使用同一强度坐标轴。基线由 5 点 SG 平滑后的输入估计，扣减仍使用原始谱。

In [ ]:
plot_baseline_fit(prepared, input_config.bad_bands, baseline_xlim)


## 原始谱与基线去除谱


In [ ]:
plot_baseline_comparison(prepared, input_config.bad_bands, baseline_xlim)


## 网络输入三通道


## 文件夹全部原始谱与拟合基线

逐条绘制文件夹内的原始谱（蓝线）及其实际扣除的拟合基线（橙线），每个子图对应一条谱。


In [ ]:
if folder_dir is None:
    print("输入是单个样本，跳过文件夹绘图；把 INPUT_PATH 改成文件夹即可。")
else:
    folder_xlim = None
    if FOLDER_DISPLAY_MIN is not None and FOLDER_DISPLAY_MAX is not None:
        folder_xlim = (FOLDER_DISPLAY_MIN, FOLDER_DISPLAY_MAX)
    plot_folder_spectra(folder_dir, input_config, build_values, folder_xlim)